In [18]:
# 모든 프레임에 박스가 표시되도록 수정된 버전
import torch
import torchvision.transforms as transforms
from torchvision.models import detection
import cv2
import numpy as np
import matplotlib.pyplot as plt
import glob
import os

class PeopleNetVideoAnalyzer:
    def __init__(self):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = None
        self.confidence_threshold = 0.5
        
        print(f"🚀 사용 중인 디바이스: {self.device}")
        self._load_model()
    
    def _load_model(self):
        try:
            self.model = detection.fasterrcnn_resnet50_fpn(pretrained=True)
            self.model.to(self.device)
            self.model.eval()
            print("✅ 모델 로드 완료!")
        except Exception as e:
            print(f"❌ 모델 로드 실패: {e}")
    
    def get_video_files(self):
        video_files = []
        extensions = ['*.mp4', '*.webm', '*.mkv', '*.avi']
        for ext in extensions:
            video_files.extend(glob.glob(f'./downloads/{ext}'))
        return video_files
    
    def detect_people(self, frame):
        transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.ToTensor(),
        ])
        
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        input_tensor = transform(frame_rgb).unsqueeze(0).to(self.device)
        
        with torch.no_grad():
            predictions = self.model(input_tensor)
        
        people_detections = []
        for i, (box, score, label) in enumerate(zip(
            predictions[0]['boxes'].cpu().numpy(),
            predictions[0]['scores'].cpu().numpy(),
            predictions[0]['labels'].cpu().numpy()
        )):
            if label == 1 and score > self.confidence_threshold:
                x1, y1, x2, y2 = box.astype(int)
                people_detections.append({
                    'bbox': [x1, y1, x2, y2],
                    'confidence': float(score),
                    'class': 'person'
                })
        
        return people_detections
    
    def draw_detections(self, frame, detections):
        for detection in detections:
            x1, y1, x2, y2 = detection['bbox']
            confidence = detection['confidence']
            
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            label = f"Person: {confidence:.2f}"
            cv2.putText(frame, label, (x1, y1-10), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        
        return frame
    
    def process_video(self, video_path, output_path=None, sample_frames=True):
        """수정된 버전: 모든 프레임에 박스 표시"""
        cap = cv2.VideoCapture(video_path)
        
        if not cap.isOpened():
            print("❌ 영상을 열 수 없습니다!")
            return None
        
        fps = int(cap.get(cv2.CAP_PROP_FPS))
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        print(f"📹 영상 정보: {width}x{height}, {fps}fps, {total_frames/fps:.1f}초")
        
        # 출력 영상 설정
        if output_path:
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
            print(f"💾 결과 영상 저장 위치: {output_path}")
        
        frame_count = 0
        people_counts = []
        sample_results = []
        
        # 샘플링 간격 설정
        if sample_frames:
            sample_interval = 5  # 5프레임마다 새로 검출
        else:
            sample_interval = 1  # 모든 프레임 검출
        
        print(f"🔄 영상 처리 중... (매 {sample_interval}프레임마다 검출)")
        
        current_detections = []  # 현재 검출 결과 저장
        
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            # 새로 검출할 프레임인지 확인
            if frame_count % sample_interval == 0:
                current_detections = self.detect_people(frame)
                people_count = len(current_detections)
                people_counts.append(people_count)
                print(f"프레임 {frame_count}: {people_count}명 검출")
            
            # 모든 프레임에 현재 검출 결과 적용
            processed_frame = self.draw_detections(frame.copy(), current_detections)
            cv2.putText(processed_frame, f"People Count: {len(current_detections)}", 
                       (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
            
            # 샘플 결과 저장
            if len(sample_results) < 5 and frame_count % (sample_interval * 10) == 0:
                sample_results.append({
                    'frame': processed_frame,
                    'frame_number': frame_count,
                    'people_count': len(current_detections)
                })
            
            # 결과 영상에 저장
            if output_path:
                out.write(processed_frame)
            
            frame_count += 1
        
        cap.release()
        if output_path:
            out.release()
            print(f"🎉 결과 영상 저장 완료: {output_path}")
        
        avg_people = np.mean(people_counts) if people_counts else 0
        max_people = max(people_counts) if people_counts else 0
        
        results = {
            'average_people_count': avg_people,
            'max_people_count': max_people,
            'people_counts_timeline': people_counts,
            'sample_frames': sample_results,
            'output_video_path': output_path if output_path else None
        }
        
        print(f"✅ 처리 완료! 평균: {avg_people:.1f}명, 최대: {max_people}명")
        return results
    
    def visualize_results(self, results):
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        
        axes[0].plot(results['people_counts_timeline'])
        axes[0].set_title('시간대별 검출된 인원 수')
        axes[0].set_xlabel('프레임 (샘플)')
        axes[0].set_ylabel('인원 수')
        axes[0].grid(True)
        
        for i, sample in enumerate(results['sample_frames'][:2]):
            if i < 2:
                frame_rgb = cv2.cvtColor(sample['frame'], cv2.COLOR_BGR2RGB)
                axes[i+1].imshow(frame_rgb)
                axes[i+1].set_title(f'프레임 {sample["frame_number"]}: {sample["people_count"]}명')
                axes[i+1].axis('off')
        
        plt.tight_layout()
        plt.show()

print("🎯 수정완료! 이제 모든 프레임에 박스가 계속 표시됩니다!")

🎯 수정완료! 이제 모든 프레임에 박스가 계속 표시됩니다!


In [ ]:
# 이제 결과 영상 저장 가능!
analyzer = PeopleNetVideoAnalyzer()

video_files = analyzer.get_video_files()
if video_files:
    results = analyzer.process_video(
        video_files[0], 
        output_path='./people_detected_result.mp4',  # 이제 작동함!
        sample_frames=True
    )
    
    if results:
        print(f"🎉 완료! 결과 영상: {results['output_video_path']}")
        analyzer.visualize_results(results)

🚀 사용 중인 디바이스: cuda
✅ 모델 로드 완료!
📹 영상 정보: 640x360, 29fps, 50.2초
💾 결과 영상 저장 위치: ./people_detected_result.mp4
🔄 영상 처리 중... (매 5프레임마다 검출)
프레임 0: 32명 검출
프레임 5: 32명 검출
프레임 10: 33명 검출
프레임 15: 33명 검출
프레임 20: 33명 검출
프레임 25: 37명 검출
프레임 30: 35명 검출
프레임 35: 36명 검출
프레임 40: 38명 검출
프레임 45: 39명 검출
프레임 50: 38명 검출
프레임 55: 45명 검출
프레임 60: 43명 검출
프레임 65: 42명 검출
프레임 70: 38명 검출
프레임 75: 40명 검출
프레임 80: 40명 검출
프레임 85: 38명 검출
프레임 90: 35명 검출
프레임 95: 36명 검출
프레임 100: 35명 검출
프레임 105: 39명 검출
프레임 110: 38명 검출
프레임 115: 37명 검출
프레임 120: 35명 검출
프레임 125: 39명 검출
프레임 130: 37명 검출
프레임 135: 39명 검출
프레임 140: 39명 검출
프레임 145: 39명 검출
프레임 150: 37명 검출
프레임 155: 38명 검출
프레임 160: 41명 검출
프레임 165: 41명 검출
프레임 170: 36명 검출
프레임 175: 37명 검출
프레임 180: 39명 검출
프레임 185: 41명 검출
프레임 190: 38명 검출
프레임 195: 40명 검출
프레임 200: 40명 검출
프레임 205: 39명 검출
프레임 210: 42명 검출
프레임 215: 40명 검출
프레임 220: 40명 검출
프레임 225: 44명 검출
프레임 230: 40명 검출
프레임 235: 39명 검출
프레임 240: 38명 검출
프레임 245: 40명 검출
프레임 250: 43명 검출
프레임 255: 39명 검출
프레임 260: 41명 검출
프레임 265: 39명 검출
프레임 270: 41명 검출
프레임 275: 